# Week 03 - Regression and Classification Basics

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 04  
**Estimated time:** 3-4 hours

This notebook is self-contained. You will train your first supervised models, compare them against simple baselines, and add numeric results to the course benchmark table.


## What You Are Building

This week has five required functions:

1. `split_scale_data(X, y, task_type, test_size, random_state)` - split data correctly and fit the scaler only on training data.
2. `evaluate_regression_models(models, X_train, X_test, y_train, y_test)` - compare regression models numerically.
3. `evaluate_classification_models(models, X_train, X_test, y_train, y_test)` - compare classification models numerically.
4. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append model results to the reusable benchmark format.
5. `plot_decision_boundary(clf, X, y, feature_names, title)` - visualise a 2D classifier.

The goal is not to memorise model APIs. The goal is to practise a supervised-learning workflow: split, preprocess without leakage, fit, evaluate against baselines, and record comparable numeric results.


In [ ]:
# Imports - keep this cell stable
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes, load_wine
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.25


## Datasets

We use built-in sklearn datasets so the notebook works without network access.

- Regression: Diabetes, predicting a quantitative disease-progression target.
- Classification: Wine, predicting cultivar class from chemical measurements.


In [ ]:
# Regression dataset
regression_data = load_diabetes(as_frame=True)
X_reg = regression_data.data
Y_REG_NAME = "disease_progression"
y_reg = regression_data.target.rename(Y_REG_NAME)

print("Regression data:", X_reg.shape)
display(X_reg.head())
display(y_reg.head())


In [ ]:
# Classification dataset
classification_data = load_wine(as_frame=True)
X_clf = classification_data.data
Y_CLF_NAME = "wine_class"
y_clf = classification_data.target.rename(Y_CLF_NAME)
class_names = classification_data.target_names

print("Classification data:", X_clf.shape)
display(X_clf.head())
display(y_clf.value_counts().sort_index().rename(index=dict(enumerate(class_names))))


## Task 1 - Split and Scale Without Leakage

Implement `split_scale_data(X, y, task_type, test_size, random_state)`.

Rules:

- Split before scaling.
- Fit `StandardScaler` only on `X_train`.
- Transform both `X_train` and `X_test` with that fitted scaler.
- Use stratification only for classification.
- Return `(X_train_scaled, X_test_scaled, y_train, y_test, scaler)`.

The returned scaled feature arrays may be numpy arrays or dataframes, but the row counts must match the targets.


In [ ]:
def split_scale_data(
    X: pd.DataFrame,
    y: pd.Series,
    task_type: str,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Split data and scale features without leaking test-set information."""
    stratify = y if task_type == "classification" else None
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

    return X_train_scaled, X_test_scaled, y_train.copy(), y_test.copy(), scaler


In [ ]:
# Self-check: Task 1
Xr_train, Xr_test, yr_train, yr_test, reg_scaler = split_scale_data(X_reg, y_reg, task_type="regression")
Xc_train, Xc_test, yc_train, yc_test, clf_scaler = split_scale_data(X_clf, y_clf, task_type="classification")

assert len(Xr_train) == len(yr_train)
assert len(Xr_test) == len(yr_test)
assert len(Xc_train) == len(yc_train)
assert len(Xc_test) == len(yc_test)
assert hasattr(reg_scaler, "mean_"), "Return the fitted scaler"
assert np.allclose(np.asarray(Xr_train).mean(axis=0), 0, atol=1e-7), "Training features should be centered after scaling"
assert not np.allclose(np.asarray(Xr_test).mean(axis=0), 0, atol=1e-2), "Do not fit the scaler on the test set"

# Stratification should preserve all classes in train and test.
assert set(yc_train.unique()) == set(y_clf.unique())
assert set(yc_test.unique()) == set(y_clf.unique())

print("Task 1 passed")


## Task 2 - Regression Model Evaluation

Implement `evaluate_regression_models(models, X_train, X_test, y_train, y_test)`.

Input: a dictionary mapping model names to sklearn estimators.  
Output: a dataframe with columns:

- `model`
- `rmse`
- `mae`
- `r2`
- `fit_time_sec`

Fit each model on the training set and evaluate on the test set. Sort by `rmse` ascending.


In [ ]:
def evaluate_regression_models(
    models: dict,
    X_train,
    X_test,
    y_train,
    y_test,
) -> pd.DataFrame:
    """Fit regression models and return comparable test metrics."""
    rows = []

    for name, model in models.items():
        start = perf_counter()
        model.fit(X_train, y_train)
        fit_time = perf_counter() - start
        predictions = model.predict(X_test)
        rmse = float(np.sqrt(mean_squared_error(y_test, predictions)))

        rows.append(
            {
                "model": name,
                "rmse": rmse,
                "mae": float(mean_absolute_error(y_test, predictions)),
                "r2": float(r2_score(y_test, predictions)),
                "fit_time_sec": float(fit_time),
            }
        )

    columns = ["model", "rmse", "mae", "r2", "fit_time_sec"]
    return pd.DataFrame(rows, columns=columns).sort_values("rmse").reset_index(drop=True)


In [ ]:
# Self-check: Task 2
regression_models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear_regression": LinearRegression(),
    "ridge_alpha_1": Ridge(alpha=1.0),
    "lasso_alpha_0.01": Lasso(alpha=0.01, max_iter=10000),
}

regression_results = evaluate_regression_models(regression_models, Xr_train, Xr_test, yr_train, yr_test)

assert isinstance(regression_results, pd.DataFrame)
assert list(regression_results.columns) == ["model", "rmse", "mae", "r2", "fit_time_sec"]
assert len(regression_results) == len(regression_models)
assert regression_results["rmse"].is_monotonic_increasing
assert regression_results[["rmse", "mae", "r2", "fit_time_sec"]].notna().all().all()
assert regression_results.loc[regression_results["model"] == "dummy_mean", "r2"].iloc[0] <= 0.1

print("Task 2 passed")
regression_results


## Guided Analysis: Regression Residuals

Use the best regression model from the table above and inspect residuals. This is not a required function; focus on what the plot tells you.


In [ ]:
best_regression_name = regression_results.iloc[0]["model"]
best_regression_model = regression_models[best_regression_name]
best_regression_model.fit(Xr_train, yr_train)
yr_pred = best_regression_model.predict(Xr_test)
residuals = yr_test - yr_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(yr_pred, residuals, alpha=0.7)
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residual")
axes[0].set_title(f"Residuals: {best_regression_name}")

sns.histplot(residuals, kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.show()


## Task 3 - Classification Model Evaluation

Implement `evaluate_classification_models(models, X_train, X_test, y_train, y_test)`.

Output columns:

- `model`
- `accuracy`
- `f1_macro`
- `fit_time_sec`

Fit each model on the training set and evaluate on the test set. Sort by `f1_macro` descending.


In [ ]:
def evaluate_classification_models(
    models: dict,
    X_train,
    X_test,
    y_train,
    y_test,
) -> pd.DataFrame:
    """Fit classifiers and return comparable test metrics."""
    rows = []

    for name, model in models.items():
        start = perf_counter()
        model.fit(X_train, y_train)
        fit_time = perf_counter() - start
        predictions = model.predict(X_test)
        rows.append(
            {
                "model": name,
                "accuracy": float(accuracy_score(y_test, predictions)),
                "f1_macro": float(f1_score(y_test, predictions, average="macro")),
                "fit_time_sec": float(fit_time),
            }
        )

    columns = ["model", "accuracy", "f1_macro", "fit_time_sec"]
    return pd.DataFrame(rows, columns=columns).sort_values("f1_macro", ascending=False).reset_index(drop=True)


In [ ]:
# Self-check: Task 3
classification_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(max_iter=2000),
    "knn_5": KNeighborsClassifier(n_neighbors=5),
}

classification_results = evaluate_classification_models(classification_models, Xc_train, Xc_test, yc_train, yc_test)

assert isinstance(classification_results, pd.DataFrame)
assert list(classification_results.columns) == ["model", "accuracy", "f1_macro", "fit_time_sec"]
assert len(classification_results) == len(classification_models)
assert classification_results["f1_macro"].is_monotonic_decreasing
assert classification_results[["accuracy", "f1_macro", "fit_time_sec"]].notna().all().all()
assert classification_results.loc[classification_results["model"] == "logistic_regression", "accuracy"].iloc[0] > 0.85

print("Task 3 passed")
classification_results


## Guided Analysis: Confusion Matrix

Inspect the best classifier's mistakes.


In [ ]:
best_classifier_name = classification_results.iloc[0]["model"]
best_classifier = classification_models[best_classifier_name]
best_classifier.fit(Xc_train, yc_train)
yc_pred = best_classifier.predict(Xc_test)

cm = confusion_matrix(yc_test, yc_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix: {best_classifier_name}")
plt.tight_layout()
plt.show()

print(classification_report(yc_test, yc_pred, target_names=class_names))


## Task 4 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

This function converts a model-results table into the reusable long benchmark format.

Output columns:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

The input `results` will contain a `model` column plus metric columns such as `rmse`, `r2`, `accuracy`, or `f1_macro`. Do not include timing columns as metrics.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert a model-results table to the cumulative benchmark long format."""
    metric_columns = [column for column in results.columns if column not in ["model", "fit_time_sec"]]
    rows = []

    for _, result in results.iterrows():
        for metric in metric_columns:
            rows.append(
                {
                    "week": week,
                    "dataset": dataset,
                    "task_type": task_type,
                    "target": target,
                    "model": result["model"],
                    "metric": metric,
                    "score": float(result[metric]),
                    "split": split,
                    "notes": "scaled features; held-out test evaluation",
                }
            )

    columns = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Self-check: Task 4
regression_benchmark = make_benchmark_long(
    regression_results,
    week="W03",
    dataset="Diabetes",
    task_type="regression",
    target=Y_REG_NAME,
    split="75_25_random_state_42",
)
classification_benchmark = make_benchmark_long(
    classification_results,
    week="W03",
    dataset="Wine",
    task_type="classification",
    target=Y_CLF_NAME,
    split="75_25_stratified_random_state_42",
)

benchmark_long = pd.concat([regression_benchmark, classification_benchmark], ignore_index=True)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert set(benchmark_long["task_type"]) == {"regression", "classification"}
assert "fit_time_sec" not in set(benchmark_long["metric"])
assert benchmark_long["score"].notna().all()

print("Task 4 passed")
benchmark_long.head(10)


## Benchmark Wide View

The long table is what you append to. The wide table is what you inspect: datasets and metrics are rows, models are columns.


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Task 5 - Decision Boundary

Implement `plot_decision_boundary(clf, X, y, feature_names, title)`.

Use only two features. Fit the classifier inside the function, draw the predicted regions on a grid, and overlay the training points. Return `(fig, ax)`.


In [ ]:
def plot_decision_boundary(clf, X, y, feature_names: list[str], title: str):
    """Fit a 2D classifier and plot its decision boundary."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf.fit(X_scaled, y)

    x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
    y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = clf.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="viridis")
    scatter = ax.scatter(
        X_scaled[:, 0],
        X_scaled[:, 1],
        c=y,
        cmap="viridis",
        edgecolor="black",
        linewidth=0.4,
    )
    ax.set_title(title)
    ax.set_xlabel(f"{feature_names[0]} (scaled)")
    ax.set_ylabel(f"{feature_names[1]} (scaled)")
    ax.legend(*scatter.legend_elements(), title="class", loc="best")
    fig.tight_layout()
    return fig, ax


In [ ]:
# Self-check: Task 5
feature_pair = ["alcohol", "flavanoids"]
X_pair = X_clf[feature_pair]
fig, ax = plot_decision_boundary(
    LogisticRegression(max_iter=2000),
    X_pair,
    y_clf,
    feature_names=feature_pair,
    title="Wine logistic regression boundary",
)

assert fig is not None and ax is not None
assert ax.get_title()
plt.show()
print("Task 5 passed")


## Reflection

Answer briefly, but concretely.

1. Which regression model performed best on Diabetes? Did it beat the dummy baseline by a meaningful margin?
2. Which classification model performed best on Wine? Did accuracy and macro-F1 tell the same story?
3. Looking at `benchmark_wide`, what does the table make easy to compare? What does it still hide?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Add Another Dataset
Add the built-in Breast Cancer dataset as a second classification row in the benchmark table.

### Track B - Add Another Model
Add `RandomForestClassifier` or `RandomForestRegressor` to the relevant benchmark. You will study trees soon; for now, treat it as a black-box comparison.

### Track C - Robustness
Modify `make_benchmark_long` so it accepts an optional `notes` dictionary keyed by model name.


In [ ]:
# Optional challenge workspace
# Your code here
